# Lab04 Task 2-2: Manual Post-Training Static Quantization

In this notebook, we will try to manually quantize the pretrained model.

<font color="red">**Only add or modify code between `YOUR CODE START` and `YOUR CODE END`. Don’t change anything outside of these markers.**</font>

In [1]:
##### YOUR CODE START #####

# Please fill in your student id here.
student_id = "314510196"

##### YOUR CODE END #####

### Library Import

The libraries you need for this practice are listed below. You can add more if you think they’re necessary. If you’re not sure whether a library is allowed, ask TA in the FB group.

In [2]:
import os
import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
import copy
from resnet20_int8 import (
    QuantizedTensor,
    QuantizedCifarResNet,
    QuantizeLayer,
    QuantizedConv2d,
    QuantizedConvReLU2d,
    QuantizedReLU,
    QuantizedLinear,
    QuantizedAdaptiveAvgPool2d,
    QuantizedAdd,
    QuantizedFlatten,
)
import matplotlib

##### YOUR CODE START #####
import types
# Do you need any additional libraries? If not, you can leave this block empty.
# For this task, you must attempt to manually quantize the model. Therefore, using any libraries that perform automatic quantization or calculate scale/zero-point values is prohibited.

##### YOUR CODE END #####

### Device

If you have GPU available, you should see "cuda" in the following cell.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: %s" % device)

Using device: cuda


### Dataset

In this lab, we will use CIFAR-10 dataset. CIFAR-10 is a widely used image classification dataset consisting of 60,000 color images at 32×32 resolution. It has 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck), with 50,000 training images and 10,000 test images. Due to its small size and balanced categories, CIFAR-10 is commonly used for benchmarking machine learning and computer vision models.

CIFAR-10 has both a training set and a test set. Post-training static quantization requires a small subset of the training set for calibration. On the other hand, manually quantizing the convolutional layer weights is a data-free process. The test set is only used at the end to evaluate the final result.

In [4]:
# Load training & test set

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128,
                                         shuffle=False)
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                       download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                         shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


### Load Model

In this lab, you do not need to train a model from scratch. We will use a pretrained ResNet20 model instead. ResNet20 is a popular deep learning model for image classification. Its key feature is the use of skip (residual) connections, which make training deep networks easier and more stable.
The code below loads the pre-trained model and evaluates its accuracy on the test set, which should be <font color="red">**92.60%**</font>. Please use this model for the subsequent tasks. <font color="red">**Designing and training your own model is not allowed.**</font>

In [5]:
model = torch.hub.load('chenyaofo/pytorch-cifar-models', 
                       'cifar10_resnet20', pretrained=True).to(device)
model.eval()

Using cache found in /nashome/NVL4/msedalab/m314510196/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

In [6]:
def test_acc(model_test):
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model_test(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f'Accuracy on CIFAR-10 test set: {acc:.2f}%')
    return acc
    
_ = test_acc(model)

Accuracy on CIFAR-10 test set: 92.59%


### Manually Quantizing the Model

While PyTorch's FX graph mode is very convenient, it abstracts away the details of how quantized weights are actually calculated. Therefore, in this section, you will attempt to manually quantize the model.

To pass this lab, the test accuracy of your manually quantized model must be higher than <font color="red">**90.00%.**</font>

<font color="red">**Please be aware of the following rules. Violating them will result in a score of zero for this section:**</font>

1. Your modifications to the model are strictly limited to populating the parameters of the `QuantizedCifarResNet` model. Any other operations, including but not limited to retraining, or changing the model architecture, are forbidden.

2. You must explicitly show your calculation process. The use of any functions that automatically compute scale / zero_point or gather statistics is prohibited. (The pre-defined observer in the previous task is prohibited, but it is allowed to use `torch.max` and `torch.min`, or define an observer on your own.) Also, you must not directly assign numerical values without demonstrating how they were derived.

### Introduction to QuantizedCifarResNet

`QuantizedCifarResNet` is a modified version of the standard `CifarResNet` architecture, specifically adapted for **integer-only inference**. Unlike the original `CifarResNet` which performs computations using 32-bit floating-point (FP32) numbers, this quantized version primarily uses 8-bit integer arithmetic (`int8` for weights, `uint8` for activations) for most of its operations. This significantly reduces model size and can lead to faster inference speeds on hardware with specialized integer instruction support.

The key differences arise from replacing standard PyTorch layers (`nn.Conv2d`, `nn.ReLU`, `nn.Linear`, etc.) with custom-defined quantized layer equivalents. These custom layers require specific **quantization parameters** (scale and zero-point) to map the integer values back to the approximate floating-point range, ensuring the model maintains reasonable accuracy. Data between these layers is passed using a `QuantizedTensor` wrapper object, which bundles the `uint8` tensor data with its corresponding `scale` and `zero_point`.

Here's a breakdown of the custom quantized layers used in this implementation and the parameters they typically require **after initialization** (usually set via methods like `set_..._params` or `set_..._weight` after calibration):

1.  **`QuantizeLayer`**:
    * **Role**: The entry point, converts the initial `float32` input tensor into a `QuantizedTensor` (`uint8`).
    * **Parameters Needed**:
        * `output_scale (float)`: The scale factor for the output activation.
        * `output_zero_point (int)`: The zero-point for the output activation.

2.  **`QuantizedConv2d`** (Used for `conv2` in BasicBlock and `downsample`):
    * **Role**: Performs 2D convolution using `int8` weights and `uint8` activations, producing a `uint8` output. Uses `fp32` bias.
    * **Parameters Needed**:
        * `weight_int8 (torch.Tensor[int8])`: The quantized weights (typically obtained after fusing BatchNorm).
        * `weight_scale (torch.Tensor[float32])`: The **per-channel** scale factor for the weights.
        * `weight_zero_point (torch.Tensor[int32])`: The **per-channel** zero-point for the weights.
        * `bias_fp32 (torch.Tensor[float32])`: The fused `float32` bias term.
        * `output_scale (float)`: The scale factor for the output activation.
        * `output_zero_point (int)`: The zero-point for the output activation.

3.  **`QuantizedConvReLU2d`** (Used for `conv1` in BasicBlock and the first `conv1` of the network):
    * **Role**: Fuses `Conv2d` and `ReLU` operations. Similar to `QuantizedConv2d` but applies ReLU before the final requantization step. Uses `fp32` bias. **The output zero-point is implicitly 0 due to ReLU.**
    * **Parameters Needed**:
        * `weight_int8 (torch.Tensor[int8])`
        * `weight_scale (torch.Tensor[float32])` (**Per-channel**)
        * `weight_zero_point (torch.Tensor[int32])` (**Per-channel**)
        * `bias_fp32 (torch.Tensor[float32])`
        * `output_scale (float)` (Output zero-point is fixed to 0 internally).

4.  **`QuantizedReLU`**:
    * **Role**: Applies ReLU activation directly on the `uint8` tensor by clamping values below the input `zero_point`.
    * **Parameters Needed**: None (It's stateless and uses the parameters from the input `QuantizedTensor`).

5.  **`QuantizedAdd`**:
    * **Role**: Performs element-wise addition of two `QuantizedTensor` inputs (requiring dequantization, float addition, and requantization). Used for residual connections.
    * **Parameters Needed**:
        * `output_scale (float)`: The scale factor for the resulting summed activation.
        * `output_zero_point (int)`: The zero-point for the resulting summed activation.

6.  **`QuantizedAdaptiveAvgPool2d`**:
    * **Role**: Performs adaptive average pooling on the `uint8` tensor.
    * **Parameters Needed**: None (Stateless, passes through the input scale/zero-point after integer averaging).

7.  **`QuantizedFlatten`**:
    * **Role**: Flattens the `uint8` tensor while preserving scale/zero-point.
    * **Parameters Needed**: None (Stateless).

8.  **`QuantizedLinear`** (Used as the final `fc` layer):
    * **Role**: Performs a linear transformation using `int8` weights and `uint8` input, producing a `float32` output (common for the final classification layer). Uses `fp32` bias.
    * **Parameters Needed**:
        * `weight_int8 (torch.Tensor[int8])`
        * `weight_scale (torch.Tensor[float32])` (**Per-channel/output feature**)
        * `weight_zero_point (torch.Tensor[int32])` (**Per-channel/output feature**)
        * `bias_fp32 (torch.Tensor[float32])`

You can find more details of `QuantizedCifarResNet` in `resnet20_int8.py`

In [7]:
model_manual = QuantizedCifarResNet().to(device)
model_manual.eval()

QuantizedCifarResNet(
  (quant): QuantizeLayer(output_scale=1.000000, output_zero_point=0)
  (conv1): QuantizedConvReLU2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=1, bias=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): QuantizedConvReLU2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=1, bias=True)
      (conv2): QuantizedConv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=1, bias=True)
      (relu): QuantizedReLU(Quantized ReLU (uint8 clamp at zero_point))
      (add): QuantizedAdd()
    )
    (1): BasicBlock(
      (conv1): QuantizedConvReLU2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=1, bias=True)
      (conv2): QuantizedConv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=1, bias=True)
      (relu): QuantizedReLU(Quantized ReLU (uint8 clamp at zero_point))
      (add): QuantizedAdd()
    )
    (2): BasicBlock(
      (conv1): QuantizedConvReLU2d(16, 16, kerne

#### 1. Prepare the Model

Let's manually quantize the model step-by-step. We'll start with the preparation phase. Although PyTorch's FX graph mode offers `prepare_fx` to automatically insert observers and fuse layers (e.g., `Conv2d`, `BatchNorm2d`, `ReLU`), we need to do this manually here. So, we'll define our own observer now and insert it into the FP32 model. Layer fusion mainly concerns weight recalculation, so we'll handle that later in step three. Below is an example of defining a min/max observer and inserting it into the initial `bn1` layer.

In [8]:
class Observer:
    def __init__(self, ema_decay=0.99, use_ema=False):
        # 初始化最小值/最大值
        self.min_val = float('inf')
        self.max_val = float('-inf')
        self.ema_decay = ema_decay
        self.use_ema = use_ema  # True → 使用滑動平均，False → 真實 min/max 範圍

    def __call__(self, module: nn.Module, inputs: tuple, output: torch.Tensor):
        """forward hook 呼叫時執行，更新觀察到的 min/max"""
        if isinstance(output, (list, tuple)):
            output = output[0]
        batch_min = output.detach().min().item()
        batch_max = output.detach().max().item()

        if self.use_ema:
            # 使用 EMA 平滑更新
            if self.min_val == float('inf'):
                self.min_val, self.max_val = batch_min, batch_max
            else:
                self.min_val = (self.min_val * self.ema_decay) + (batch_min * (1 - self.ema_decay))
                self.max_val = (self.max_val * self.ema_decay) + (batch_max * (1 - self.ema_decay))
        else:
            # 使用實際最小值/最大值（不平滑）
            self.min_val = min(self.min_val, batch_min)
            self.max_val = max(self.max_val, batch_max)

    def get_min_max(self) -> tuple[float, float]:
        """回傳觀察到的整體最小與最大值"""
        return self.min_val, self.max_val

    def reset(self):
        """重設觀察結果"""
        self.min_val = float('inf')
        self.max_val = float('-inf')


# ==============================================================
# 2. 重新定義 BasicBlock.forward，以分離 relu1 / relu2
# ==============================================================
def new_basic_block_forward(self, x):
    """修正版 forward：使用獨立的 relu1 與 relu2"""
    identity = x

    # conv1 → bn1 → relu1
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu1(out)

    # conv2 → bn2
    out = self.conv2(out)
    out = self.bn2(out)

    # 若存在 downsample
    if self.downsample is not None:
        identity = self.downsample(x)

    # 加上殘差
    out += identity

    # relu2（但我們要在 ReLU 之前取得 "add_out"）
    out = self.relu2(out)
    return out


# ==============================================================
# 3. 建立觀察器並掛上 hook
# ==============================================================
model_prepared = copy.deepcopy(model)
observers = {}

# --- 頂層 ReLU (stem) ---
model_prepared.relu.inplace = False
observers['stem.relu_out'] = Observer()
model_prepared.relu.register_forward_hook(observers['stem.relu_out'])

# --- 輸入觀察器（手動呼叫） ---
observers['input'] = Observer()

# --- 每個 BasicBlock ---
layers = [model_prepared.layer1, model_prepared.layer2, model_prepared.layer3]

for i, layer in enumerate(layers):
    for j, block in enumerate(layer.children()):
        block_name = f'layer{i+1}.{j}'

        # 1. 建立獨立的 ReLU
        block.relu1 = nn.ReLU(inplace=False)
        block.relu2 = nn.ReLU(inplace=False)
        del block.relu  # 刪除原本共用的 relu

        # 2. 替換 forward
        block.forward = types.MethodType(new_basic_block_forward, block)

        # 3. 註冊各層觀察器
        # (1) conv1 → relu1 後的輸出（activation > 0）
        observers[f'{block_name}.relu1_out'] = Observer()
        block.relu1.register_forward_hook(observers[f'{block_name}.relu1_out'])

        # (2) conv2 → bn2 後的輸出（允許負值）
        observers[f'{block_name}.bn2_out'] = Observer()
        block.bn2.register_forward_hook(observers[f'{block_name}.bn2_out'])

        # (3) residual add 後（ReLU 前）的輸出 → QuantizedAdd 專用
        observers[f'{block_name}.add_out'] = Observer()
        def _pre_relu2_hook(mod, inp, name=f'{block_name}.add_out'):
            # inp[0] 即為 add 後、ReLU 前的張量
            observers[name](mod, None, inp[0])
        block.relu2.register_forward_pre_hook(_pre_relu2_hook)

        # (4) downsample 分支（若存在）
        if block.downsample is not None and len(block.downsample) >= 2:
            observers[f'{block_name}.downsample_out'] = Observer()
            # downsample = Sequential(Conv, BN)
            block.downsample[1].register_forward_hook(observers[f'{block_name}.downsample_out'])

##### YOUR CODE END #####

#### 2. Calibrate the Model

Next, we need to calibrate the model. This step is quite similar to the process when using PyTorch's FX graph mode. It simply involves feeding data from the training set (or a representative subset of it) through the model we prepared earlier (the one with observers attached). Please note: <font color="red">**it is crucial not to use the test set data for calibration.**</font>

In [9]:
##### YOUR CODE START #####

print("Starting calibration...")
model_prepared.eval()
model_prepared.to(device)

with torch.no_grad():
    for batch_idx, (images, _) in enumerate(trainloader):
        images = images.to(device)

        # 手動觀察輸入張量 (QuantizeLayer)
        observers['input'](None, None, images)

        # Forward pass → 觸發所有 hooks 收集 activation 範圍
        _ = model_prepared(images)

        # 可視情況加速，只取前 N 個 batch 代表性統計
        if batch_idx >= 128:
            break

print("Calibration finished.\nExample observed ranges (min/max):")

# 範例輸出 (命名與 Step 1 一致)
print(f"  Input: {observers['input'].get_min_max()}")
print(f"  stem.relu_out: {observers['stem.relu_out'].get_min_max()}")
print(f"  layer1.0.relu1_out: {observers['layer1.0.relu1_out'].get_min_max()}")
print(f"  layer1.0.bn2_out: {observers['layer1.0.bn2_out'].get_min_max()}")
print(f"  layer1.0.add_out: {observers['layer1.0.add_out'].get_min_max()}")

##### YOUR CODE END #####


Starting calibration...
Calibration finished.
Example observed ranges (min/max):
  Input: (-2.429065704345703, 2.7537312507629395)
  stem.relu_out: (0.0, 3.833252191543579)
  layer1.0.relu1_out: (0.0, 3.424288511276245)
  layer1.0.bn2_out: (-3.342231512069702, 3.9921176433563232)
  layer1.0.add_out: (-3.342231512069702, 5.585373401641846)


#### 3. Convert the Model

Finally, we need to populate the `QuantizedCifarResNet` with its parameters. You will need to iterate through all layers in the quantized model and set the required parameters (such as quantized weights, scales, zero-points, and biases) based on their type. The necessary data should be obtained from the original model and the observers inserted previously. Additionally, note that consecutive `Conv2d`, `BatchNorm2d`, and `ReLU` layers in the original model have been fused into corresponding single layers in `QuantizedCifarResNet`. You must adjust the `Conv2d` weights and biases according to the parameters of the corresponding `BatchNorm2d` layer.

In [10]:
##### YOUR CODE START #####

# ------------------------------
# Activations: uint8 affine quant
# ------------------------------
def get_quant_params_activations(observer: Observer, *, relu_output: bool = False):
    r_min, r_max = observer.get_min_max()
    # 退化防護
    if r_max <= r_min:
        r_max = r_min + 1e-6

    if relu_output:
        # ReLU 之後：min≈0 → zp=0, scale=max/255
        max_val = max(1e-6, r_max)
        scale = max_val / 255.0
        zero_point = 0
    else:
        # 一般 affine：允許負值
        scale = (r_max - r_min) / 255.0
        scale = float(max(scale, 1e-8))
        zero_point = int(round(-r_min / scale))
        zero_point = max(0, min(255, zero_point))
    return float(scale), int(zero_point)

# ------------------------------
# Fuse Conv+BN (FP32)
# ------------------------------
def fuse_conv_bn(conv: nn.Conv2d, bn: nn.BatchNorm2d):
    gamma = bn.weight
    beta  = bn.bias
    mean  = bn.running_mean
    var   = bn.running_var
    eps   = bn.eps

    W = conv.weight
    b = conv.bias if conv.bias is not None else torch.zeros_like(mean)

    sigma = torch.sqrt(var + eps)
    alpha = gamma / sigma  # per-out channel
    W_fused = W * alpha.view(-1,1,1,1)
    b_fused = (b - mean) * alpha + beta
    return W_fused, b_fused

# ------------------------------
# Weights: per-channel symmetric int8
#   s_c = max(|w_c|)/127, zp_c = 0
# ------------------------------
def get_weight_symm_perchannel_params(W_fused: torch.Tensor):
    Cout = W_fused.shape[0]
    W_flat = W_fused.view(Cout, -1)
    max_abs = W_flat.abs().max(dim=1).values + 1e-8
    scale = (max_abs / 127.0).to(torch.float32)              # [Cout]
    zero_point = torch.zeros_like(scale, dtype=torch.int32)   # [Cout], symmetric
    return scale, zero_point

def quantize_weights_symm_perchannel(W_fused: torch.Tensor, scale: torch.Tensor):
    Cout = W_fused.shape[0]
    W_flat = W_fused.view(Cout, -1)
    W_q = torch.round(W_flat / scale.unsqueeze(1)).clamp(-127, 127).to(torch.int8)
    return W_q.view_as(W_fused)

# ------------------------------
# 名稱映射 & 取對應的 FP32 模組 / 觀察器
# ------------------------------
def get_fp32_conv_bn_and_obs(name: str):
    """
    回傳 (conv_module, bn_module, out_observer, is_relu_fused)
    is_relu_fused=True 表示該量化層為 ConvReLU2d，輸出 zp=0。
    """
    if name == "conv1":  # stem
        conv = model.conv1
        bn   = model.bn1
        obs  = observers['stem.relu_out']          # ReLU 後
        return conv, bn, obs, True

    parts = name.split('.')  # 如: ['layer1','0','conv1'] or ['layer1','0','downsample','0']
    if "downsample" in parts:
        # downsample 內通常只有 conv+bn
        layer = getattr(model, parts[0])[int(parts[1])]
        conv = layer.downsample[0]
        bn   = layer.downsample[1]
        obs  = observers[f'{parts[0]}.{parts[1]}.downsample_out']  # BN 後 (允許負)
        return conv, bn, obs, False

    # block conv1/conv2
    layer = getattr(model, parts[0])[int(parts[1])]
    if parts[2] == "conv1":
        conv = layer.conv1
        bn   = layer.bn1
        obs  = observers[f'{parts[0]}.{parts[1]}.relu1_out']       # ReLU 後
        return conv, bn, obs, True
    else:
        conv = layer.conv2
        bn   = layer.bn2
        obs  = observers[f'{parts[0]}.{parts[1]}.bn2_out']         # BN 後 (允許負)
        return conv, bn, obs, False

# ------------------------------
# 迭代量化模型並填入參數
# ------------------------------
# 假設你的量化骨架叫 model_manual，且類型如下（如 resnet20_int8.py）：
#   QuantizeLayer, QuantizedConv2d, QuantizedConvReLU2d, QuantizedAdd, QuantizedLinear
for name, module in model_manual.named_modules():

    # 入口量化：輸入張量 → uint8
    if module.__class__.__name__ == "QuantizeLayer":
        s, z = get_quant_params_activations(observers['input'], relu_output=False)
        print(f"[Setup] QuantizeLayer scale={s:.6f}, zp={z}")
        # 你的 API 可能叫 set_output_params / set_output_quant_params，依你的檔案對應
        if hasattr(module, "set_output_quant_params"):
            module.set_output_quant_params(s, z)
        else:
            module.set_output_params(s, z)

    # 卷積（含 ConvReLU2d）
    elif module.__class__.__name__ in ("QuantizedConv2d", "QuantizedConvReLU2d"):
        conv, bn, obs_out, is_relu_fused = get_fp32_conv_bn_and_obs(name)

        # (1) BN 融合
        W_fused, b_fused = fuse_conv_bn(conv, bn)

        # (2) 權重量化參數（per-channel symmetric int8）
        w_scale, w_zp = get_weight_symm_perchannel_params(W_fused)   # zp=0 vec
        w_int8 = quantize_weights_symm_perchannel(W_fused, w_scale)

        # (3) 輸出 activation 量化參數
        s_out, z_out = get_quant_params_activations(obs_out, relu_output=is_relu_fused)

        # (4) 寫回量化層
        if hasattr(module, "set_weight_quant_params"):
            module.set_weight_quant_params(w_scale.to(device), w_zp.to(device))
        else:
            module.weight_scale = w_scale.to(device)
            module.weight_zero_point = w_zp.to(device)

        if hasattr(module, "set_int8_weight"):
            module.set_int8_weight(w_int8.to(device))
        else:
            module.weight_int8 = w_int8.to(device)

        if hasattr(module, "set_fp32_bias"):
            module.set_fp32_bias(b_fused.to(device))
        else:
            module.bias_fp32 = b_fused.to(device)

        if module.__class__.__name__ == "QuantizedConvReLU2d":
            # ReLU fused → zp 固定 0，介面通常只收 scale
            if hasattr(module, "set_output_quant_params"):
                module.set_output_quant_params(float(s_out))
            else:
                module.output_scale = float(s_out)
                module.output_zero_point = 0
        else:
            if hasattr(module, "set_output_quant_params"):
                module.set_output_quant_params(float(s_out), int(z_out))
            else:
                module.output_scale = float(s_out)
                module.output_zero_point = int(z_out)

    # 殘差加總 → 使用 add_out（ReLU 前）
    elif module.__class__.__name__ == "QuantizedAdd":
        parts = name.split('.')  # e.g., layer1.0.add
        obs = observers[f'{parts[0]}.{parts[1]}.add_out']
        s, z = get_quant_params_activations(obs, relu_output=False)
        if hasattr(module, "set_output_quant_params"):
            module.set_output_quant_params(float(s), int(z))
        else:
            module.output_scale = float(s)
            module.output_zero_point = int(z)

    # 最後 Linear（fc）：權重 per-out-channel 對稱 int8、bias FP32；輸出常為 FP32
    elif module.__class__.__name__ == "QuantizedLinear":
        fc = model.fc
        W_fused = fc.weight
        b_fused = fc.bias

        w_scale, w_zp = get_weight_symm_perchannel_params(W_fused)
        w_int8 = quantize_weights_symm_perchannel(W_fused, w_scale)

        if hasattr(module, "set_weight_quant_params"):
            module.set_weight_quant_params(w_scale.to(device), w_zp.to(device))
        else:
            module.weight_scale = w_scale.to(device)
            module.weight_zero_point = w_zp.to(device)

        if hasattr(module, "set_int8_weight"):
            module.set_int8_weight(w_int8.to(device))
        else:
            module.weight_int8 = w_int8.to(device)

        if hasattr(module, "set_fp32_bias"):
            module.set_fp32_bias(b_fused.to(device))
        else:
            module.bias_fp32 = b_fused.to(device)

##### YOUR CODE END #####


[Setup] QuantizeLayer scale=0.020325, zp=120


In [11]:
# Let's see the result.
acc = test_acc(model_manual)

print("\n===========================================\n")

if acc < 90.0:
    print("Oh no! Your test accuracy is too low!")
else:
    print("Congratulations! You've achieved the goal of this task. Remember to save your model!")
    print("You can also try increasing accuracy further to earn a higher score!")

Accuracy on CIFAR-10 test set: 92.49%


Congratulations! You've achieved the goal of this task. Remember to save your model!
You can also try increasing accuracy further to earn a higher score!


### Save Model

You can use the code below to save your model as `[student_id]_quantization.pt`, where `[student_id]` is replaced by your student ID in the first cell of this notebook.

In [12]:
file_name = student_id + "_quantization.pt"
# Save model.state_dict() instead of the entire model.
torch.save(model_manual.state_dict(), file_name)
print("Your model is saved to \"" + file_name + "\".")

Your model is saved to "314510196_quantization.pt".


### Final Check

TA has provided check_quantization.py for students to check if their models can pass the tests. <font color="red">**Please make sure to check it before submission.**</font>

In [13]:
!python check_quantization.py --path {file_name}

Files already downloaded and verified
Congratulations! You've achieved the goals of this task.
Your model's test accuracy is 92.49%.
